# Trend Tracker — Notebook 03: Insights Generation

Runs the full analytical pipeline on the enriched corpus, from TF-IDF and NMF
topic discovery through LLM synthesis, verification, packaging, and DOCX report
generation.

**Run order:** `01_token_preprocess` → `02_semantic_enrichment` → `03_insights_generation`

| Step | What | Key outputs |
|------|------|-------------|
| 1 | Build TF-IDF matrices | in-memory for Steps 2–4 |
| 2 | Quality checkpoint 2 | `quality/quality_cp2.json` |
| 3 | Category TF-IDF | `analysis/category_tfidf.csv` |
| 4 | NMF topic discovery | `analysis/nmf_topics.csv`, `analysis/project_topic_bridge.csv` |
| 5 | LLM topic labeling | `analysis/llm_topic_labels.json` |
| 6 | Synthesis | `analysis/llm_synthesis_*.txt` |
| 7 | Structured insight extraction | `insights/insights_candidates.json` |
| 8 | Topic verification | (updates insights in-memory) |
| 9 | Evidence tables | `insights/insights_flat.csv`, `insights/insight_topic_support_candidates.csv` |
| 10 | Packaging, dedupe & topline | (updates curated insights in-memory) |
| 11 | Final outputs & manifests | `reports/trend_tracker_report.docx`, `insights/insights_structured.json` |

---
## Setup and Configuration

In [1]:
import sys
import json
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

import pandas as pd
import numpy as np

ROOT = Path(".")
sys.path.insert(0, str(ROOT))

import importlib.util as _ilu, sys as _sys
_u = next(Path(".").glob("utils*.py"))
if _u.stem != "utils":
    _s = _ilu.spec_from_file_location("utils", _u); _m = _ilu.module_from_spec(_s); _s.loader.exec_module(_m); _sys.modules["utils"] = _m
    
from utils import (
    load_cfg,
    write_json,
    artifact_meta,
    build_run_output_path,
    get_run_date,
    canonicalize_filter_spec,
    get_filter_fields_key,
    get_run_id,
    apply_filters,
    ensure_warning_file,
    append_warning,
    get_llm_client,
    start_stage_manifest,
    finalize_stage_manifest,
    build_pipeline_manifest,
    tokens_to_str,
    make_vec,
    add_bin,
    group_key,
    build_project_topic_bridge,
    quality_report,
    cat_tfidf_slice,
    nmf_one,
    build_input,
    _label_with_retry,
    _norm_group_value,
    _safe_topic_id,
    build_topic_lines,
    _call_with_retry,
    synthesize_one_group,
    strip_json_fences,
    normalize_insight,
    build_bridge_lookup,
    build_label_index,
    build_verified_insight_tables,
    apply_deterministic_packaging,
    dedupe_packaged_insights,
    assign_topline_sections_simple,
    build_structured_from_curated,
    build_looker_project_url,
    build_packaged_report_docx,
    project_insight_for_saved_candidates,
    _verify_insight_list,
    resolve_params_path,
)

CFG_PATH = resolve_params_path()
CFG      = load_cfg(CFG_PATH)

try:
    MANIFEST_CFG_PATH = str(CFG_PATH.relative_to(ROOT))
except ValueError:
    MANIFEST_CFG_PATH = str(CFG_PATH)
ct       = CFG["tfidf"]
client   = get_llm_client()

# Load the enriched corpus produced by NB02. To use the unenriched baseline
# instead, change the path to 05_filtered.parquet.
raw_df = pd.read_parquet(ROOT / "OUTPUTS/prepared/06_enriched.parquet")

# Stopwords for the quality gate at Step 2. Loaded from params.yaml so the
# list can be extended without touching notebook code.
STOPWORDS = CFG["quality"]["stopword_violation_list"]

print(f"Loaded {len(raw_df):,} projects  |  {raw_df.shape[1]} columns")

Loaded 2,244,916 projects  |  183 columns


---
## Parameters

All runtime configuration is resolved here from `params.yaml`. Edit `params.yaml`
to change behaviour; do not hardcode values in downstream cells.

In [2]:
# ── New groupby fields ────────────────────────────────────────────────────────
# ── Funded/expired x metro, excluding live ────────────
expiration_dt = pd.to_datetime(raw_df.get("expiration_date"), errors="coerce")

status = pd.Series(
    np.select(
        [
            raw_df["funded_date"].notna(),
            expiration_dt.le(pd.Timestamp.today().normalize()),
        ],
        ["Funded", "Expired"],
        default=pd.NA,
    ),
    index=raw_df.index,
)

raw_df["funding_status_x_metro"] = (
    status.astype("string") + " | " +
    raw_df["metro_type_at_time_of_posting"].fillna("Unknown").astype("string")
).where(status.notna(), pd.NA)

# ── Derived groupby field: EFS x grade band ──────────────────────────────────
rural = raw_df["school_is_underserved_rural"].eq("Yes")
race  = raw_df["school_is_historically_underrepresented_race"].eq("Yes")
inc   = raw_df["school_is_low_income"].eq("Yes")

efs = pd.Series(
    np.select(
        [race & inc, rural, race, inc],
        ["Race+Inc", "Rural", "Race", "Inc"],
        default="NonEFS",
    ),
    index=raw_df.index,
)

raw_df["efs_x_grade_band"] = (
    efs.astype("string") + " | " +
    raw_df["grade_band"].fillna("Unknown").astype("string")
)

#raw_df["funding_status_x_metro"].value_counts(dropna=False)
#raw_df["efs_x_grade_band"].value_counts(dropna=False)

# ── Waterfall groupby fields from injected taxonomy tokens ────────────────────
# For each priority list, pick the first listed token present in the project's
# enriched token list; projects with none get "__NA__" so they can be excluded
# via exclude_groups or a filter in NB03.

import re

_PREFIX_RE = re.compile(r"^__(sensitive_context|subject|industry|context|framing)_")
_SUFFIX_RE = re.compile(r"_(context|framing)__$")

def short_name(tok):
    """Strip namespace prefix and trailing _context/_framing suffix for display."""
    return _SUFFIX_RE.sub("__", _PREFIX_RE.sub("__", tok))

def first_present(tokens, priority):
    s = set(tokens)
    match = next((t for t in priority if t in s), None)
    return short_name(match) if match else "__NA__"

WATERFALLS = {
    "specialty_subject": [
        "__subject_coding_computer_science__",
        "__subject_ai_literacy_foundations__",
        "__subject_restorative_practices_conflict_resolution__",
        "__subject_cte_credentials_certifications__",
        "__subject_multilingual_language_access__",
        "__subject_entrepreneurship_business_skills__",
        "__subject_ai_accessibility_assistive_use__",
        "__subject_science_inquiry_lab_learning__",
        "__subject_math_manipulatives_quantitative_learning__",
        "__subject_cooking_culinary_life_skills__",
        "__subject_financial_literacy_personal_finance__",
        "__subject_robotics_makerspace_3d_printing__",
        "__subject_climate_environmental_science_learning__",
        "__subject_nutrition_literacy_health_education__",
        "__subject_home_language_family_literacy__",
        "__subject_sel_curriculum_skill_building__",
        "__subject_assistive_adaptive_technology__",
        "__subject_food_systems_agriculture_learning__",
        "__subject_engineering_design_maker_learning__",
        "__subject_employability_durable_skills__",
        "__subject_english_language_development__",
        "__subject_phonics_decodable_reading_intervention__",
    ],
    "industry": [
        "__industry_agriculture_food_systems_careers__",
        "__industry_public_safety_civic_careers__",
        "__industry_green_jobs_sustainability_careers__",
        "__industry_transportation_logistics_aviation__",
        "__industry_robotics_engineering_careers__",
        "__industry_business_entrepreneurship_careers__",
        "__industry_health_careers__",
        "__industry_media_creative_industries__",
    ],
    "safety_justice": [
        "__sensitive_context_gang_violence_exposure_context__",
        "__sensitive_context_justice_involved_family_context__",
        "__sensitive_context_community_violence_safety_context__",
        "__context_alternative_school_justice_adjacent_context__",
        "__context_direct_attendance_absence_terms__",
        "__context_discipline_exclusion_absence__",
        "__context_mentoring_protective_support__",
    ],
    "future_framing": [
        "__framing_ai_literacy_responsible_use__",
        "__framing_ambitious_programmatic_ask__",
        "__framing_applied_production_real_world_execution__",
        "__framing_career_exploration_pathways__",
        "__framing_career_identity_exposure__",
        "__framing_coding_computational_thinking_foundation__",
        "__framing_future_preparation__",
        "__framing_opportunity_expansion__",
        "__framing_technical_pathway_skill_building__",
    ],
    "framing_context": [
        "__framing_belonging_identity_affirmation__",
        "__framing_catch_up_remediation__",
        "__framing_chronic_scarcity__",
        "__framing_disability_accommodation_access__",
        "__framing_event_news_disaster_anchoring__",
        "__framing_experiential_hands_on_learning__",
        "__framing_expressive_creative_production__",
        "__framing_home_extension_family_colearning__",
        "__framing_inclusion_accommodation_framing__",
        "__framing_rural_scarcity_dispersion__",
        "__framing_safety_protection_language__",
        "__framing_stigma_avoidance_social_confidence__",
        "__framing_student_voice_belonging_identity__",
    ],
    "current_pressures": [
        "__context_ai_policy_integrity_classroom_management__",
        "__context_cellphone_management_distraction__",        
        "__context_chronic_absenteeism_reengagement__",        
        "__context_climate_heat_air_quality_disruption__",     
        "__context_esser_cliff_budget_pressure__",             
        "__context_school_safety_security_protocols__",        
        "__context_student_mental_health_anxiety_stress__",    
        "__context_teacher_staffing_substitute_shortage__",    
    ],
}

for field, priority in WATERFALLS.items():
    raw_df[field] = raw_df["tokens"].apply(lambda ts, p=priority: first_present(ts, p))
    n = (raw_df[field] != "__NA__").sum()
    print(f"{field:28s}  {n:>6,} tagged  ({n/len(raw_df):.1%})")
'''
# Strip taxonomy tokens (anything starting with "__") now that waterfalls are assigned
before = raw_df["tokens"].str.len().sum()
raw_df["tokens"] = raw_df["tokens"].apply(lambda ts: [t for t in ts if not t.startswith("__")])
after = raw_df["tokens"].str.len().sum()
print(f"\nStripped {before - after:,} taxonomy tokens ({before:,} → {after:,})")'''

specialty_subject             534,880 tagged  (23.8%)
industry                      250,567 tagged  (11.2%)
safety_justice                133,666 tagged  (6.0%)
future_framing                329,245 tagged  (14.7%)
framing_context               613,154 tagged  (27.3%)
current_pressures             149,465 tagged  (6.7%)

Stripped 6,882,577 taxonomy tokens (138,092,769 → 131,210,192)


In [3]:
# ── Analysis scope ────────────────────────────────────────────────────────────
GROUPBY_FIELD  = CFG["analysis"]["group_by"]
FILTER_LOGIC   = CFG["analysis"].get("filter_logic", "and")
FILTERS        = CFG["analysis"].get("filters", [])
EXCLUDE_GROUPS = CFG["analysis"]["exclude_groups"]
REVIEW_GROUP   = CFG["analysis"]["review_group"]

# Apply declarative filters first so all corpus-scope checks run on the
# right population.
df, filter_summary = apply_filters(raw_df, FILTER_LOGIC, FILTERS)
if filter_summary["no_rows_after_filter"]:
    raise ValueError("No rows remain after applying analysis filters — check params.yaml.")

# ── v1 feature guards ─────────────────────────────────────────────────────────
# Binned/time-slice mode: topic identity and downstream traceability (bridge
# table, label index, source_topics) are keyed by group + topic_id only.
# Adding a bin dimension would cause silent topic-id collisions across bins
# within the same group. Guard here until topic identity is bin-aware.
if CFG["analysis"].get("bins", []):
    raise ValueError(
        "analysis.bins is non-empty: binned/time-slice mode is not supported "
        "in v1. Topic identity and source_topic traceability are not bin-aware. "
        "Set analysis.bins: [] to run without time bucketing."
    )

# review_group: restricting to a single group would bypass cross-group
# synthesis and require conditional output handling that is not wired in v1.
# Disable until the cross-group path is made optional.
if REVIEW_GROUP is not None:
    raise ValueError(
        f"analysis.review_group is set to {REVIEW_GROUP!r}: single-group review "
        "mode is not supported in v1. Set analysis.review_group: null to run "
        "against all groups."
    )

# ── Group scope: apply exclude_groups once, here, for the whole pipeline ──────
# Removing excluded groups from df means every downstream step (TF-IDF, NMF,
# labeling, synthesis) automatically excludes them. No per-step filtering needed.
if EXCLUDE_GROUPS:
    n_before = len(df)
    df = df[
        ~df[GROUPBY_FIELD].astype(str).isin([str(g) for g in EXCLUDE_GROUPS])
    ].reset_index(drop=True)
    if df.empty:
        raise ValueError(
            f"No rows remain after excluding groups {EXCLUDE_GROUPS} — "
            "check analysis.exclude_groups in params.yaml."
        )
    print(f"exclude_groups: {n_before - len(df):,} rows removed "
          f"({len(EXCLUDE_GROUPS)} group(s): {EXCLUDE_GROUPS})")

# ── Run identity ───────────────────────────────────────────────────────────────
# RUN_DATE and RUN_ID are assigned BEFORE OUT() is defined because OUT() closes
# over them. Reversing this order causes a NameError at the OUT() call site.
FILTER_SPEC       = canonicalize_filter_spec(FILTER_LOGIC, FILTERS)
FILTER_FIELDS_KEY = get_filter_fields_key(FILTERS)
RUN_DATE          = get_run_date()
RUN_ID            = get_run_id(GROUPBY_FIELD, FILTER_SPEC)

def OUT(subdir, fname):
    return build_run_output_path(
        subdir=subdir, fname=fname,
        groupby_field=GROUPBY_FIELD, run_date=RUN_DATE, run_id=RUN_ID,
    )

# ── Analysis description and topic/slice settings ─────────────────────────────
GROUP_DESCRIPTION = CFG["analysis"]["group_descriptions"].get(
    GROUPBY_FIELD,
    f"groups defined by '{GROUPBY_FIELD}' in DonorsChoose data",
)
MIN_SHARED    = CFG["analysis"]["nmf_min_shared"]
MIN_COVERAGE  = CFG["analysis"]["min_coverage"]
BASE_N_COMPONENTS = CFG["nmf"]["n_components"]
CAT_TFIDF_TOP_N   = CFG["analysis"]["cat_tfidf_top_n"]

SLICE_RULES   = CFG["analysis"]["slice_rules"]
VERIFY_CFG    = CFG["analysis"]["verification"]
PACKAGING_CFG = CFG["analysis"]["packaging"]
DEDUPE_CFG    = CFG["analysis"]["dedupe"]

# group_project_counts is computed on the scoped df (post-filter, post-exclude)
# so small_slice_mode reflects the actual groups that will run.
group_project_counts = (
    df[[GROUPBY_FIELD, "project_id"]]
    .dropna(subset=["project_id"])
    .drop_duplicates()
    .groupby(GROUPBY_FIELD)["project_id"]
    .nunique()
)
median_group_projects = float(group_project_counts.median()) if not group_project_counts.empty else 0.0
SMALL_SLICE_MODE = median_group_projects < SLICE_RULES["small_slice_cutoff"]
SLICE_RULES      = {**SLICE_RULES, "small_slice_mode": SMALL_SLICE_MODE}

MIN_GROUP_PROJECTS_EFFECTIVE = max(
    CFG["analysis"]["min_group_projects"],
    SLICE_RULES["min_group_projects"],
)
CROSS_MIN_INSIGHTS     = SLICE_RULES["cross_min_insights"]
CROSS_MAX_INSIGHTS     = SLICE_RULES["cross_max_insights"]
PER_GROUP_MIN_INSIGHTS = SLICE_RULES["per_group_min_insights"]
PER_GROUP_MAX_INSIGHTS = SLICE_RULES["per_group_max_insights"]

# ── LLM settings ──────────────────────────────────────────────────────────────
N_REPRESENTATIVE          = CFG["llm"]["n_representative_snippets"]
TOP_TERMS_IN_PROMPT       = CFG["llm"]["top_terms_in_prompt"]
SYNTHESIS_TOP_TERMS_COUNT = CFG["llm"]["synthesis_top_terms_count"]
# MAX_RETRIES is passed to all LLM call sites so retry behaviour is tunable
# from params.yaml without touching notebook code.
MAX_RETRIES = CFG["llm"]["max_retries"]

MODEL_LABELING  = CFG["models"]["labeling"]
MODEL_SYNTHESIS = CFG["models"]["synthesis"]
MODEL_VERIFY    = CFG["models"]["verify"]

MAX_WORKERS          = CFG["analysis"]["synthesis_max_workers"]
LABELING_MAX_WORKERS = CFG["analysis"]["labeling_max_workers"]

# ── Output settings ────────────────────────────────────────────────────────────
LOOKER_BASE_URL     = CFG["output"]["looker_base_url"]
LOOKER_FILTER_FIELD = CFG["output"]["looker_filter_field"]
LOOKER_FIELDS       = CFG["output"]["looker_fields"]
LOOKER_LIMIT        = int(CFG["output"].get("looker_limit", 500))
LOOKER_ID_LIMIT     = int(CFG["output"].get("looker_id_limit", 100))

# CSV_MAX_IDS_PER_INSIGHT: ranked project IDs stored per insight row in
# insights_flat.csv. Passed to build_verified_insight_tables().
CSV_MAX_IDS_PER_INSIGHT = int(CFG["output"]["csv_max_ids_per_insight"])

# MAIN_MIN_VERIFICATION_RATIO: eligibility gate for main-section topline
# placement. Passed to assign_topline_sections_simple().
MAIN_MIN_VERIFICATION_RATIO = PACKAGING_CFG["main_min_verification_ratio"]

# bins are guarded out above; group_cols is always [GROUPBY_FIELD] in v1.
group_cols = [GROUPBY_FIELD]

# ── Warnings, filter provenance, stage manifest ────────────────────────────────
WARNINGS_PATH       = OUT("metadata", "warnings_03.jsonl")
FILTER_SPEC_PATH    = OUT("metadata", "filter_spec.json")
FILTER_SUMMARY_PATH = OUT("metadata", "filter_summary.json")
COPIED_CONFIG_PATH  = OUT("metadata", CFG_PATH.name)

ensure_warning_file(WARNINGS_PATH)
write_json(FILTER_SPEC_PATH, {
    "schema_version": "v1", "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD, "filter_fields_key": FILTER_FIELDS_KEY,
    "filter_logic": FILTER_LOGIC, "filters": FILTERS,
})
write_json(FILTER_SUMMARY_PATH, {
    "schema_version": "v1", "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD, **filter_summary,
})
shutil.copy2(CFG_PATH, COPIED_CONFIG_PATH)

STAGE_MANIFEST = start_stage_manifest(
    stage_name="03_insights_generation",
    notebook_file="03_insights_generation_v1.ipynb",
    config_path=MANIFEST_CFG_PATH,
    
    run_id=RUN_ID,
    group_by_field=GROUPBY_FIELD,
    filter_fields_key=FILTER_FIELDS_KEY,
)

print(f"RUN_ID            = {RUN_ID}")
print(f"GROUPBY_FIELD     = {GROUPBY_FIELD!r}")
print(f"FILTER_FIELDS_KEY = {FILTER_FIELDS_KEY}")
print(f"Filtered rows     = {len(df):,} / {len(raw_df):,}")
print(f"small_slice_mode  = {SMALL_SLICE_MODE}  (median group = {median_group_projects:.1f})")
print(f"Output root       = OUTPUTS/runs/{GROUPBY_FIELD}/{RUN_DATE}/{RUN_ID}/")

exclude_groups: 688,961 rows removed (1 group(s): ['__NA__'])
RUN_ID            = 20260514_210306_safety_justice_0c79fb23
GROUPBY_FIELD     = 'safety_justice'
FILTER_FIELDS_KEY = expiration_date
Filtered rows     = 33,165 / 2,244,916
small_slice_mode  = False  (median group = 3729.0)
Output root       = OUTPUTS/runs/safety_justice/2026-05-14/20260514_210306_safety_justice_0c79fb23/


---
## Step 1 — Build TF-IDF Matrices

Fits one TF-IDF vectorizer per n-gram range on the full filtered corpus and
keeps the resulting sparse matrices in memory. All downstream steps reuse these
objects rather than refitting.

Trigrams are built when `ngrams.max_n >= 3` in `params.yaml`.

In [4]:
docs = df["tokens"].apply(tokens_to_str).tolist()

# Build one matrix per n-gram range. X_unigram_bigram is the primary matrix
# for category TF-IDF and NMF; the others are retained for debugging and
# future analytical passes.
specs = {
    "X_unigram":        (1, 1),
    "X_bigram":         (2, 2),
    "X_unigram_bigram": (1, 2),
}
if CFG["ngrams"]["max_n"] >= 3:
    specs["X_trigram"] = (3, 3)

matrices, vecs = {}, {}
for name, rng in specs.items():
    vec = make_vec(ct["min_df"], ct["max_df"], rng)
    matrices[name] = vec.fit_transform(docs)
    vecs[name] = vec
    sz, nnz = matrices[name].shape, matrices[name].nnz
    print(f"  {name:20s}: shape={sz}  sparsity={1 - nnz / (sz[0] * sz[1]):.3f}")

# Spot-check first features to catch bad filtering or token drift.
for name, vec in vecs.items():
    print(f"  {name:20s} first feats → {vec.get_feature_names_out()[:10].tolist()}")

  X_unigram           : shape=(33165, 5647)  sparsity=0.990
  X_bigram            : shape=(33165, 32130)  sparsity=0.999
  X_unigram_bigram    : shape=(33165, 37777)  sparsity=0.998
  X_trigram           : shape=(33165, 6484)  sparsity=0.999
  X_unigram            first feats → ['aac', 'abc', 'ability', 'able', 'above', 'absence', 'absent', 'absenteeism', 'absolute', 'absolutely']
  X_bigram             first feats → ['ability able', 'ability access', 'ability attend', 'ability become', 'ability bring', 'ability build', 'ability calm', 'ability choose', 'ability communicate', 'ability concentrate']
  X_unigram_bigram     first feats → ['aac', 'abc', 'ability', 'ability able', 'ability access', 'ability attend', 'ability become', 'ability bring', 'ability build', 'ability calm']
  X_trigram            first feats → ['ability concentrate participate', 'ability express themselves', 'ability focus class', 'ability focus engage', 'ability focus participate', 'ability focus regulate', 'abili

---
## Step 2 — Quality Checkpoint 2

Gate before LLM calls. Review before proceeding:

- **No stopword violations** — if terms from `quality.stopword_violation_list`
  appear in the top-200 vocab, re-run NB01 with tighter frequency thresholds.
- **Reasonable token distribution** — `p50` should be in the 20–60 range.
- **Matrix sparsity** — very high sparsity on the bigram matrix suggests
  `tfidf.min_df` may be too restrictive.

In [5]:
# Pass the configured stopword list so the gate reflects params.yaml rather
# than the HARD_STOPWORDS fallback in utils.py.
qr2 = quality_report(
    df, label="cp2",
    matrices=matrices,
    save_path=OUT("quality", "quality_cp2.json"),
    stopwords=STOPWORDS,
)


=======================================================  [cp2]
  Projects : 33,165
  Tok/proj : min=14  p50=53  max=80
  Vocab    : 7,051 unique tokens
  X_unigram           : shape=[33165, 5647]  sparsity=0.990
  X_bigram            : shape=[33165, 32130]  sparsity=0.999
  X_unigram_bigram    : shape=[33165, 37777]  sparsity=0.998
  X_trigram           : shape=[33165, 6484]  sparsity=0.999
  Stopwords: FAIL — ['project', 'teacher', 'grade', 'education']



---
## Step 3 — Category TF-IDF

The vectorizer is fit **once** on the full corpus; category slices are scored
by index. This avoids the string-comparison bug (identical token sets across
projects would misclassify rows) and is much faster than refitting per slice.

**Contrast** = token prevalence in this category minus prevalence outside it.

Time bins: defined in `params.yaml` under `analysis.bins`; leave empty for
the full date range.


In [6]:
# ── Category TF-IDF ────────────────────────────────────────────────────────
# Score each eligible group against the rest of the corpus using a shared matrix.

top_n = CAT_TFIDF_TOP_N
min_proj = MIN_GROUP_PROJECTS_EFFECTIVE

df_work = df.copy().reset_index(drop=True)
all_docs = df_work["tokens"].apply(tokens_to_str).tolist()

vec_cat = make_vec(ct["min_df"], ct["max_df"], tuple(ct["ngram_range"]))
X_full = vec_cat.fit_transform(all_docs)
feat = vec_cat.get_feature_names_out()
idf_vals = vec_cat.idf_

rows = []
for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj:
        continue

    kd = group_key(keys, group_cols)
    top = cat_tfidf_slice(
        sub.index,
        df_index=df_work.index,
        X_full=X_full,
        feat=feat,
        idf_vals=idf_vals,
        top_n=top_n,
    )
    for col, val in kd.items():
        top.insert(0, col, val)
    rows.append(top)

if not rows:
    raise RuntimeError(
        f"No groups met min_proj={min_proj} threshold — lower min_group_projects in params.yaml"
    )

cat_tfidf_df = pd.concat(rows, ignore_index=True)
cat_tfidf_df.to_csv(OUT("analysis", "category_tfidf.csv"), index=False)

print(f"{len(cat_tfidf_df):,} rows  |  {cat_tfidf_df[group_cols[0]].nunique()} groups")
cat_tfidf_df.head(10)

210 rows  |  7 groups


,safety_justice,token,tf,idf,tfidf,prevalence,contrast,project_count
0,__alternative_school_justice_adjacent__,justice,0.040082,3.760541,0.150729,0.532960,0.529560,1997
1,__alternative_school_justice_adjacent__,alternate,0.031374,4.320037,0.135538,0.319189,0.319121,1196
2,__alternative_school_justice_adjacent__,social justice,0.016057,5.010686,0.080458,0.158260,0.158022,593
3,__alternative_school_justice_adjacent__,racial,0.012660,5.132637,0.064981,0.124900,0.122758,468
4,__alternative_school_justice_adjacent__,alternative,0.014442,4.312559,0.062280,0.157726,0.136787,591
5,__alternative_school_justice_adjacent__,alternate seat,0.009612,5.852452,0.056253,0.068588,0.068554,257
6,__alternative_school_justice_adjacent__,racial justice,0.010068,5.580335,0.056182,0.089672,0.089570,336
7,__alternative_school_justice_adjacent__,read,0.017839,2.835707,0.050585,0.313317,0.173437,1174
8,__alternative_school_justice_adjacent__,book,0.017962,2.798597,0.050268,0.315986,0.169647,1184
9,__alternative_school_justice_adjacent__,story,0.012877,3.761972,0.048444,0.180678,0.132510,677


---
## Step 4 — NMF Topic Discovery

NMF is fit independently per group so the dominant vocabulary axis in one group
does not suppress signal in others. Topics are treated as evidence candidates
for LLM synthesis — not as stable theme definitions.

The NMF weight matrix `W` records how strongly each project loads on each topic.
Step 5 uses the top-weight projects per topic as representative snippets rather
than sampling randomly.

In [7]:
# ── Groupwise NMF topics ───────────────────────────────────────────────────
# Fit one NMF model per eligible group and keep both topic-level and project-level outputs.

cn = CFG["nmf"]
min_proj = MIN_GROUP_PROJECTS_EFFECTIVE

all_topics, all_weights = [], []
df_work = df.copy().reset_index(drop=True)

NMF_GROUPS_SKIPPED = []
NMF_GROUPS_FAILED = []

for keys, sub in df_work.groupby(group_cols, observed=True):
    kd = group_key(keys, group_cols)
    group_value = kd[GROUPBY_FIELD]

    # Skip groups that are too small to support stable topic extraction.
    if len(sub) < min_proj:
        NMF_GROUPS_SKIPPED.append(str(group_value))
        continue

    group_docs = sub["tokens"].apply(tokens_to_str).tolist()
    pids = sub["project_id"].tolist()

    try:
        topics, W, nmf_meta = nmf_one(
            group_docs,
            ct_cfg=ct,
            cn_cfg=cn,
            base_n_components=BASE_N_COMPONENTS,
            slice_rules=SLICE_RULES,
        )
    except Exception as e:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF failed for group '{group_value}'",
            context={"group": group_value, "error": str(e)},
        )
        continue

    # None return means the slice was too thin; kept separate from exceptions for QA.
    if topics is None or W is None:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF skipped group '{group_value}' because the slice was too thin",
            context={"group": group_value, **(nmf_meta or {})},
        )
        continue

    for col, val in kd.items():
        topics[col] = val
    all_topics.append(topics)

    # Persist the full W ranking so later cells can recover top projects per topic.
    for tid in range(W.shape[1]):
        order = W[:, tid].argsort()[::-1]
        for rank, idx in enumerate(order):
            all_weights.append(
                {
                    **kd,
                    "topic_id": tid,
                    "project_id": pids[idx],
                    "weight": float(W[idx, tid]),
                    "rank": rank,
                }
            )

if not all_topics:
    raise RuntimeError(
        "No groups produced NMF topics — check n_components vs retained vocab, "
        "or lower min_group_projects"
    )

topics_df = pd.concat(all_topics, ignore_index=True)
weights_df = pd.DataFrame(all_weights)

topics_df.to_csv(OUT("analysis", "nmf_topics.csv"), index=False)
weights_df.to_csv(OUT("analysis", "nmf_weights.csv"), index=False)

print(f"{len(topics_df):,} topics across {topics_df[group_cols[0]].nunique()} groups")

# Build the project-topic bridge once so downstream evidence collection can reuse it.
threshold = CFG["analysis"]["topic_assignment_threshold"]
project_topic_bridge_df = build_project_topic_bridge(
    weights_df,
    GROUPBY_FIELD,
    threshold,
)
project_topic_bridge_df.to_csv(OUT("analysis", "project_topic_bridge.csv"), index=False)
print(
    "Bridge: "
    f"{len(project_topic_bridge_df):,} project-topic assignments "
    f"(topic_share >= {threshold})"
)

topics_df.head(6)

98 topics across 7 groups
Bridge: 32,996 project-topic assignments (topic_share >= 0.3)


,topic_id,top_terms,top_weights,safety_justice
0,0,"[book, read, library, interest, social justice...","[0.9518137925048717, 0.9086914999180798, 0.740...",__alternative_school_justice_adjacent__
1,1,"[citizen individual, science ela, food study, ...","[0.2587482919269085, 0.2587482919269085, 0.258...",__alternative_school_justice_adjacent__
2,2,"[seat, alternate, option, chair, alternate sea...","[0.9152345269300212, 0.6315845773773763, 0.615...",__alternative_school_justice_adjacent__
3,3,"[background urban, program design, attend alte...","[0.34890640800103834, 0.33950546090649697, 0.3...",__alternative_school_justice_adjacent__
4,4,"[spectrum disorder, autism spectrum, workforce...","[0.30585770080856717, 0.30407069362348865, 0.3...",__alternative_school_justice_adjacent__
5,5,"[critical, theme, thinke, discussion, critical...","[0.5194473027941883, 0.5084137885391102, 0.485...",__alternative_school_justice_adjacent__


In [8]:
# ── Cross-group universal themes ───────────────────────────────────────────
# Identify term bundles that recur across groups after topic extraction.

# Select the largest group as the cross-group reference point for any
# group-specific context needed in downstream steps.
REFERENCE_GROUP = df_work.groupby(GROUPBY_FIELD).size().idxmax()

records = [(r[group_cols[0]], frozenset(r.top_terms)) for _, r in topics_df.iterrows()]

# theme_cats maps a shared term bundle to the set of groups where it recurs.
theme_cats = defaultdict(set)
for i, (ci, si) in enumerate(records):
    for cj, sj in records[i + 1:]:
        if ci == cj:
            continue
        shared = si & sj
        if len(shared) >= MIN_SHARED:
            key = frozenset(shared)
            theme_cats[key] |= {ci, cj}

rows = sorted(theme_cats.items(), key=lambda x: -len(x[1]))
seen = []
deduped = []
for terms, cats in rows:
    if len(cats) < MIN_COVERAGE:
        continue
    if not any(terms <= prior for prior in seen):
        deduped.append(
            {
                "theme": ", ".join(sorted(terms)[:5]),
                "n_groups": len(cats),
                "categories": sorted(cats),
            }
        )
        seen.append(terms)

cross_group_df = pd.DataFrame(deduped).reset_index(drop=True)
cross_group_df.to_csv(OUT("analysis", "cross_group_themes.csv"), index=False)
cross_group_df

,theme,n_groups,categories
0,"income, low, low income",6,"[__alternative_school_justice_adjacent__, __co..."
1,"health, mental, mental health",5,"[__community_violence_safety__, __direct_atten..."
2,"book, read, story",4,"[__alternative_school_justice_adjacent__, __co..."


---
## Step 5 — LLM Topic Labeling

One API call per topic using compressed input — never raw essay text at scale.
Representative snippets are selected by NMF weight (highest-loading projects),
not randomly. Parallel execution via `ThreadPoolExecutor`.

Parse failures are stored with enough metadata to debug later; failed topics are
excluded from synthesis but do not halt the run.

In [9]:
# ── LLM topic labeling ─────────────────────────────────────────────────────
# Convert raw NMF topics into human-readable labels/descriptions with retry logic.

SYSTEM = (
    "You are an NLP analyst reviewing NMF topic clusters from DonorsChoose teacher "
    "essays. Respond ONLY with a single valid JSON object. No preamble. No markdown fences.\n\n"

    "Token naming conventions you may see in topic terms:\n"
    "  __framing_[name]__ = rhetorical framing, tone, mechanism, or persuasion signal\n"
    "  __subject_[name]__ = subject/domain/content signal\n"
    "  __industry_[name]__ = workforce-development industry or skill-domain signal\n"
    "  __request_[name]__ = material/request-topology signal\n"
    "  __context_[name]__ = contextual school, community, attendance, safety, or access signal\n"
    "  __sensitive_context_[name]__ = direct sensitive-context signal; describe carefully and only when supported\n"
    "  __cat_[name]__ = legacy subject matter category token\n"
    "  __sub_[name]__ = legacy subcategory token\n"
    "These injected tokens are analyst-curated semantic signals. When they appear alongside compatible "
    "terms or snippets, treat them as important evidence about the topic's function, context, or framing, "
    "not as incidental noise. They may help distinguish superficially similar topics. However, do not use "
    "an injected token as standalone proof, and do not let it override direct contradictions in the concrete "
    "terms or representative snippets.\n"
    "Never output raw tokens such as __framing_*__, __subject_*__, __industry_*__, __request_*__, "
    "__context_*__, __sensitive_context_*__, __cat_*__, __sub_*__, or snake_case token names. "
    "Always translate such signals into plain English.\n\n"

    "Language constraints:\n"
    "- Do not use em dashes.\n"
    "- Do not use reveal-style phrasing such as 'not just X,' 'really about Y,' or 'what looks like X is Y.'\n"
    "- Do not use named places.\n\n"

    "Your job is to produce a stable canonical topic label and one dense grounded description.\n"
    "Do not optimize for cleverness, vividness, or donor-facing insight language. "
    "Optimize for specificity, repeatability, plain-English clarity, and preservation of useful evidence.\n\n"

    "Rules for proposed_label:\n"
    "- proposed_label must be a short canonical noun phrase, usually 3 to 7 words.\n"
    "- Prefer the most specific defensible mechanism, request type, intervention, "
    "classroom routine, or use case.\n"
    "- Do not try to pack every nuance into proposed_label.\n"
    "- Do not collapse topics into broad umbrella labels like technology, literacy, "
    "engagement, classroom supplies, or social-emotional learning if the evidence "
    "supports a narrower interpretation.\n"
    "- Preserve concrete signals such as named programs, pedagogies, student populations, "
    "classroom routines, and rhetorical framing when clearly supported.\n\n"

    "Rules for description:\n"
    "- description must be exactly one sentence.\n"
    "- description must be plain English and evidence-led.\n"
    "- Write the description as one grounded sentence naming the dominant evidence pattern: "
    "the concrete materials or activities, the classroom function when supported, and the population, "
    "setting, or framing signal only when clearly present.\n"
    "- Prefer concrete natural-language phrasing over abstract wording.\n"
    "- Do not write implications, recommendations, or donor-facing interpretation.\n"
    "- Do not use raw preprocessing tokens or token-like jargon in description.\n\n"

    "Rules for notes:\n"
    "- notes should usually be empty.\n"
    "- Use notes only for real edge cases.\n"
    "- If coherence_flag is mixed, briefly name the colliding subthemes.\n"
    "- If coherence_flag is redundant, briefly state the nature of the overlap.\n"
    "- If coherence_flag is unclear, briefly state why the topic is too scattered.\n"
    "- Do not restate the description in notes.\n\n"

    "coherence_flag definitions:\n"
    "  coherent   — one dominant theme, mechanism, population, or use case clearly leads, "
    "even if secondary signals are present.\n"
    "  mixed      — no single dominant theme clearly leads and two or more distinguishable "
    "subthemes are truly colliding.\n"
    "  redundant  — this topic's terms and snippets substantially duplicate another "
    "topic in the same group, not merely overlap in subject area.\n"
    "  unclear    — terms are too scattered or generic to support a defensible label.\n\n"

    "Important tie-breaker:\n"
    "- If one theme is primary and another is secondary, mark the topic coherent, not mixed, "
    "and keep notes empty unless the secondary signal is important to preserve.\n"
    "- If two phrasings are both plausible, choose the more literal, reusable, and plain-English one."
)

PROMPT = (
    f"Group field: {GROUPBY_FIELD} — {GROUP_DESCRIPTION}\n"
    f"Group value: {{group_value}}{{bin_line}}\n"
    "Topic {topic_id}\n"
    "Top unigrams : {unigrams}\n"
    "Top bigrams  : {bigrams}\n"
    "Top NMF terms: {nmf_terms}\n"
    "Representative project tokens:\n"
    "{snippets}\n\n"

    "Instructions:\n"
    "- proposed_label must be a short canonical noun phrase, usually 3 to 7 words.\n"
    "- description must be exactly one sentence in plain English.\n"
    "- Write the description as one grounded sentence naming the dominant evidence pattern: "
    "the concrete materials or activities, the classroom function when supported, and the population, "
    "setting, or framing signal only when clearly present.\n"
    "- Prefer concrete mechanism and classroom function over broad category language.\n"
    "- Translate any framing/category/subcategory token signals into normal language; never copy raw token strings.\n"
    "- If the topic appears broad, identify the narrower mechanism, use case, or population "
    "if the evidence supports it.\n"
    "- Use coherence_flag='mixed' only when no single dominant theme clearly leads.\n"
    "- If one theme is primary and another is secondary, mark the topic coherent.\n"
    "- notes should usually be empty; use notes only for mixed, redundant, or unclear topics.\n"
    "- Do not write donor-facing insights, implications, or rhetorical flourishes.\n\n"

    "Output requirements:\n"
    "- proposed_label: short and stable\n"
    "- description: exactly one dense grounded sentence\n"
    '- coherence_flag: one of "coherent", "mixed", "redundant", "unclear"\n'
    "- notes: usually empty, unless needed for edge cases\n\n"

    f"Return a JSON object with exactly these keys:\n"
    f"{GROUPBY_FIELD}, topic_id, proposed_label, description, coherence_flag, notes"
)

# Build lightweight snippet text from the top projects attached to each topic.
pid_text = df.set_index("project_id")["tokens"].apply(lambda t: " ".join(t[:40]))
topic_inputs = [
    build_input(
        t,
        weights_df=weights_df,
        pid_text=pid_text,
        groupby_field=GROUPBY_FIELD,
        n_representative=N_REPRESENTATIVE,
        top_terms_in_prompt=TOP_TERMS_IN_PROMPT,
    )
    for _, t in topics_df.iterrows()
]

topic_order = {
    (_norm_group_value(inp["group_value"]), int(inp["topic_id"])): i
    for i, inp in enumerate(topic_inputs)
}

results = []
with ThreadPoolExecutor(max_workers=LABELING_MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _label_with_retry,
            inp,
            client=client,
            model_labeling=MODEL_LABELING,
            system_prompt=SYSTEM,
            user_prompt_template=PROMPT,
            groupby_field=GROUPBY_FIELD,
            warnings_path=WARNINGS_PATH,
            max_retries=MAX_RETRIES,
        ): inp
        for inp in topic_inputs
    }
    for future in as_completed(futures):
        inp = futures[future]
        obj = future.result()
        results.append(obj)
        print(f"  {inp['group_value']} / topic {inp['topic_id']} → {obj.get('proposed_label', '?')}")

# Re-sort to the original topic order so downstream joins stay stable.
bad_sort_keys = []
for r in results:
    norm_key = (
        _norm_group_value(r.get(GROUPBY_FIELD, "")),
        _safe_topic_id(r.get("topic_id", -1)),
    )
    if norm_key not in topic_order:
        bad_sort_keys.append({"group": r.get(GROUPBY_FIELD, ""), "topic_id": r.get("topic_id", "")})

if bad_sort_keys:
    raise ValueError(
        f"LABELING_SORT_KEY_MISMATCH: {len(bad_sort_keys)} label result(s) did not match the input topic order. "
        f"Examples: {bad_sort_keys[:5]}"
    )

results = sorted(
    results,
    key=lambda r: topic_order[
        (_norm_group_value(r.get(GROUPBY_FIELD, "")), _safe_topic_id(r.get("topic_id", -1)))
    ],
)

with open(OUT("analysis", "llm_topic_labels.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)

print(f"\n{len(results)} labels saved")

  __alternative_school_justice_adjacent__ / topic 12 → Forensic science crime scene investigation kits
  __alternative_school_justice_adjacent__ / topic 8 → Alternative school attendance and behavior incentives
  __alternative_school_justice_adjacent__ / topic 6 → Alternate stand desk for focus
  __community_violence_safety__ / topic 3 → Urban students writing and reading growth
  __alternative_school_justice_adjacent__ / topic 0 → Independent library reading for reluctant readers
  __alternative_school_justice_adjacent__ / topic 2 → Flexible alternate seating options
  __alternative_school_justice_adjacent__ / topic 11 → Safe space classroom calm corner
  __alternative_school_justice_adjacent__ / topic 7 → Racial justice book and art materials
  __alternative_school_justice_adjacent__ / topic 10 → Calm corner self regulation tools
  __community_violence_safety__ / topic 1 → Youth football safety equipment and training
  __community_violence_safety__ / topic 2 → Behavioral risk and sub

In [10]:
# ── Label post-processing ──────────────────────────────────────────────────
# Keep only parseable labeling results and validate them against source topics.

parse_errors = [r for r in results if r.get("parse_error")]
if parse_errors:
    print(f"WARNING: {len(parse_errors)} topics failed JSON parse — excluded from synthesis:")
    for e in parse_errors:
        print(f"  {e.get(GROUPBY_FIELD, '?')} / topic {e.get('topic_id', '?')}")

labels_df = pd.DataFrame([r for r in results if not r.get("parse_error")])

assert GROUPBY_FIELD in labels_df.columns, (
    f"labels_df missing '{GROUPBY_FIELD}' — re-run labeling for this groupby field"
)

# Source-of-truth allowed groups come from the source topic table, not model output.
ALLOWED_GROUP_VALUES = [
    str(g)
    for g in topics_df[GROUPBY_FIELD].dropna().drop_duplicates().tolist()
]

invalid_groups = sorted(
    set(labels_df[GROUPBY_FIELD].astype(str).unique()) - set(ALLOWED_GROUP_VALUES)
)
if invalid_groups:
    raise ValueError(
        f"Invalid {GROUPBY_FIELD} values returned during labeling: {invalid_groups}"
    )

# Normalize types after validation so later joins are stable.
labels_df[GROUPBY_FIELD] = labels_df[GROUPBY_FIELD].astype(str)
labels_df["topic_id"] = labels_df["topic_id"].astype(int)

labeled_groups = set(labels_df[GROUPBY_FIELD].unique()) if not labels_df.empty else set()
LABELING_FAILED_GROUPS = sorted(set(ALLOWED_GROUP_VALUES) - labeled_groups)

labels_df.groupby("coherence_flag").size().rename("count").to_frame()
labels_df[[GROUPBY_FIELD, "topic_id", "proposed_label", "coherence_flag", "description"]]

,safety_justice,topic_id,proposed_label,coherence_flag,description
0,__alternative_school_justice_adjacent__,0,Independent library reading for reluctant readers,coherent,The essays describe building and curating a di...
1,__alternative_school_justice_adjacent__,1,Food study interdisciplinary course,coherent,Essays describe an interdisciplinary food stud...
2,__alternative_school_justice_adjacent__,2,Flexible alternate seating options,coherent,Projects request movable flexible seats such a...
3,__alternative_school_justice_adjacent__,3,Alternative school library reading support,coherent,Requests for books and library access to suppo...
4,__alternative_school_justice_adjacent__,4,Alternate assessment for autism workforce read...,coherent,Teachers describe using an alternate assessmen...
...,...,...,...,...,...
93,__mentoring_protective_support__,10,Behavior reward role-modeling incentives,coherent,Teachers use point systems and classroom rewar...
94,__mentoring_protective_support__,11,mental health counseling and coping tools,coherent,Projects provide wellness and mental health re...
95,__mentoring_protective_support__,12,College and career readiness mentoring,coherent,The projects emphasize a mentoring and learnin...
96,__mentoring_protective_support__,13,Library-based mentoring for young learners,coherent,Mentoring visits that pair teachers or mentors...


---
## Step 6 — Synthesis

Produces two synthesis passes in sequence:

1. **Cross-group synthesis** — a single LLM call over all topic lines to identify
   patterns, contrasts, framing logic, and notable boundaries across the corpus.
2. **Per-group synthesis** — one LLM call per group, run in parallel, to surface
   each group's dominant internal mechanisms.

Both passes produce plain text saved to `analysis/llm_synthesis_*.txt` and are
assembled in Step 7 as the evidence base for the structured JSON insight call.

In [11]:
# ── SYNTHESIS — cross-group + per-group loops ─────────────────────────────
# First synthesize the whole landscape, then synthesize each group in parallel.

SYNTHESIS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "Your job is to synthesize topic-level evidence into stable, decision-useful analytic findings "
    "for internal strategy work. You are not writing polished external copy and you are not doing creative interpretation. "
    "You are performing disciplined evidence grouping.\n\n"

    "Core objective:\n"
    "- Produce findings that are specific, well-grounded, and tightly tied to the supplied topic lines.\n"
    "- Identify useful classroom, student, operational, funding, or context patterns without forcing surprise or contrast.\n"
    "- Literal evidence discipline is more important than elegance. Do not invent recency, current-events framing, causality, or prevalence that the topic terms and labels do not support.\n\n"

    "Evidence rules:\n"
    "- Treat the supplied topic lines as the full evidence base for this step.\n"
    "- Every finding must be supported by explicitly named topics.\n"
    "- Use only topics that directly support the claim.\n"
    "- Separate direct evidence from interpretation. Direct evidence is what the topic labels, terms, and descriptions show. Interpretation is what role that evidence appears to play for students, classrooms, or school-day operations.\n"
    "- Do not generalize beyond the named supporting topics.\n"
    "- If evidence is borderline, omit the finding rather than stretching it.\n\n"

    "Substrate rules:\n"
    "- Food, hygiene, clothing, seating, storage, sensory tools, calm spaces, headphones, basic supplies, and generic devices are common classroom substrate.\n"
    "- Do not promote common substrate as a finding unless the supplied topics show a mechanism specific to this group or cross-group pattern.\n"
    "- If substrate appears only as generic classroom operating support, treat it as background rather than as a standalone finding.\n\n"

    "Merge rules:\n"
    "- Merge topics only when they share the same dominant mechanism, classroom function, student condition, or school-day operating pressure.\n"
    "- Do not merge topics just because they involve the same product category, school setting, or broad theme.\n"
    "- Keep access/accommodation, regulation/behavior, maintenance/operations, identity/belonging, safety/security, mental health context, and routine/workflow separate unless the evidence clearly shows they are the same pattern.\n"
    "- If one candidate finding is broader and another is a more specific evidence-grounded version, prefer the more specific one.\n\n"

    "Scope rules:\n"
    "- Mark sparse or sensitive-context findings as narrow or emerging.\n"
    "- Do not use broad language such as 'teachers are' when the evidence is concentrated in a small direct-signal slice.\n"
    "- Do not use trend language such as 'growing,' 'rising,' 'increasing,' or 'more often' unless the supplied evidence includes an explicit time comparison.\n\n"

    "Support-citation rules:\n"
    "- For every finding, include a 'Supporting topics' line.\n"
    "- Each supporting topic must be written exactly as <group_value>|<topic_id>.\n"
    "- group_value must be copied verbatim from the topic lines provided. Do not abbreviate, paraphrase, normalize spelling, or invent names.\n"
    "- Do not write labels or prose in place of the support references.\n\n"

    "Writing rules:\n"
    "- Be precise, concrete, and analytical.\n"
    "- Do not mention NMF, model, cluster, pipeline, or analytical machinery.\n"
    "- Do not write donor-facing rhetoric.\n"
    "- Do not force a 'looks like X, but is really Y' structure.\n"
    "- Do not manufacture a reader mistake or surface misread.\n"
    "- Do not use em dashes.\n"
    "- Do not use named places.\n"
    "- Do not invent a cleaner pattern than the evidence supports."
)

topic_lines = build_topic_lines(
    labels_df,
    GROUPBY_FIELD,
    top_terms_count=SYNTHESIS_TOP_TERMS_COUNT,
)

SYNTHESIS_PROMPT_CROSS_GROUP = f"""
Below is a list of topic lines discovered from teacher project request essays on DonorsChoose,
grouped by "{GROUPBY_FIELD}" ({GROUP_DESCRIPTION}).

Each line contains:
- group value
- topic number
- topic label
- coherence flag
- top terms
- one-sentence description

Topic lines:
{topic_lines}

Your task:
Synthesize the topic lines into a stable cross-group analysis that will later be used to generate
evidence-grounded insights. Preserve nuance, but be disciplined about grouping.

Follow this exact workflow:
1. Review the topic lines in the order given.
2. Identify candidate cross-group findings only when multiple topics share the same dominant mechanism,
   classroom function, student condition, or school-day operating pressure.
3. Keep distinct mechanisms separate even when they live in similar product categories.
4. Rank findings by:
   a. specificity of the shared mechanism
   b. clarity of distinction from nearby themes
   c. evidence strength in the topic labels, terms, and descriptions
   d. breadth across groups
5. Treat common classroom substrate as background unless the evidence shows a distinctive cross-group mechanism.
6. Omit weak or borderline patterns rather than padding.
7. A narrower but more distinctive pattern is preferred over a broader but generic one, provided both are well-supported.

Return plain text only, using exactly this structure and these section headings:

CROSS-GROUP FINDINGS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: <group_value>|<topic_id>; <group_value>|<topic_id>; <group_value>|<topic_id>

Finding 2
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

IMPORTANT DISTINCTIONS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

BOUNDARIES OR NON-CENTRAL SIGNALS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

Hard requirements:
- CROSS-GROUP FINDINGS: return {CROSS_MIN_INSIGHTS} to {CROSS_MAX_INSIGHTS} findings
- IMPORTANT DISTINCTIONS: return 1 to 4 findings
- BOUNDARIES OR NON-CENTRAL SIGNALS: return 0 to 3 findings
- Every finding must include a Supporting topics line
- Supporting topics must use exact <group_value>|<topic_id> format
- Do not use bullet points
- Do not skip section headings
- Do not repeat the same pattern in multiple sections
- Do not force contrast language when the evidence supports a plain pattern
- Prefer mechanism-level explanations over broad summaries like access, engagement, or support
- If a pattern is supported by only one group, it does not belong in CROSS-GROUP FINDINGS
- If a distinction is real but narrow, put it in IMPORTANT DISTINCTIONS rather than inflating it into a cross-group signature
- A boundary or non-central signal may become important if it prevents overgeneralization, distinguishes a sparse emerging topic, or clarifies what the category should not be used to claim
- If a signal is sparse, sensitive, or narrow, say so in Scope
- Do not use trend language unless explicit time-comparison evidence is supplied
- Do not use named places
- Do not use em dashes

Write like a rigorous internal analyst, not like a speechwriter.
""".strip()

PER_GROUP_INSTRUCTIONS = """
Your task:
Identify the strongest, most decision-useful patterns within this single group.

Follow this exact workflow:
1. Review the topic lines in the order given.
2. Identify the dominant subthemes in the group.
3. Separate nearby themes unless they clearly share the same dominant mechanism, classroom function, student condition, or operating pressure.
4. Prefer narrower evidence-grounded findings over broad category summaries.
5. Treat common classroom substrate as background unless the group evidence shows a distinctive mechanism.
6. Omit weak or redundant findings rather than padding.

Return plain text only, using exactly this structure and these section headings:

CORE GROUP FINDINGS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: <group_value>|<topic_id>; <group_value>|<topic_id>

Finding 2
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

IMPORTANT INTERNAL DISTINCTIONS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

BOUNDARIES OR NON-CENTRAL SIGNALS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

Hard requirements:
- CORE GROUP FINDINGS: return 2 to 5 findings
- IMPORTANT INTERNAL DISTINCTIONS: return 0 to 3 findings
- BOUNDARIES OR NON-CENTRAL SIGNALS: return 0 to 2 findings
- Every finding must include a Supporting topics line
- Supporting topics must use exact <group_value>|<topic_id> format
- Do not use bullet points
- Do not skip section headings
- Do not repeat the same idea across multiple sections
- Use mixed topics carefully; do not let one mixed topic dominate a whole finding unless the collision itself is the finding
- If one theme is primary and another is secondary, keep the finding centered on the primary theme
- Do not give generic summaries of the group label
- Do not force a reveal, contrast, or looks-like-X-but-is-Y structure
- Do not use trend language unless explicit time-comparison evidence is supplied
- Do not use named places
- Do not use em dashes
- If the signal is sparse, sensitive, or narrow, say so in Scope

Write like a rigorous internal analyst, not like a speechwriter.
""".strip()

synthesis_cross = _call_with_retry(
    SYNTHESIS_PROMPT_CROSS_GROUP,
    client=client,
    model_name=MODEL_SYNTHESIS,
    system_prompt=SYNTHESIS_SYSTEM,
    max_retries=MAX_RETRIES,
)
if synthesis_cross is None:
    append_warning(
        WARNINGS_PATH,
        "03_insights_generation",
        "SYNTHESIS_CROSS_GROUP_FAILED",
        "Cross-group synthesis failed",
        context={},
    )
    raise RuntimeError("Cross-group synthesis failed; stopping before downstream cells.")

with open(OUT("analysis", "llm_synthesis_cross_group.txt"), "w", encoding="utf-8") as f:
    f.write(synthesis_cross)

# Excluded groups were removed from df in the Parameters cell; they cannot
# appear in ALLOWED_GROUP_VALUES. Filter only for label coverage.
groups = [
    g for g in ALLOWED_GROUP_VALUES
    if g in set(labels_df[GROUPBY_FIELD].unique())
]

per_group_results = {}
SYNTHESIS_FAILED_GROUPS = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            synthesize_one_group,
            g,
            labels_df=labels_df,
            groupby_field=GROUPBY_FIELD,
            group_description=GROUP_DESCRIPTION,
            per_group_instructions=PER_GROUP_INSTRUCTIONS,
            client=client,
            model_name=MODEL_SYNTHESIS,
            system_prompt=SYNTHESIS_SYSTEM,
            warnings_path=WARNINGS_PATH,
            outpath_func=OUT,
            max_retries=MAX_RETRIES,
        ): g
        for g in groups
    }
    for future in as_completed(futures):
        submitted_group = futures[future]
        try:
            group, result = future.result()
        except Exception as e:
            SYNTHESIS_FAILED_GROUPS.append(str(submitted_group))
            append_warning(
                WARNINGS_PATH,
                "03_insights_generation",
                "SYNTHESIS_GROUP_FAILED",
                f"Synthesis failed for group '{submitted_group}'",
                context={"group": submitted_group, "error": str(e)},
            )
            print(f"FAILED: {submitted_group} | error: {e}")
            continue

        if result is not None:
            per_group_results[group] = result
            print(f"Done: {group}")
        else:
            SYNTHESIS_FAILED_GROUPS.append(str(group))
            print(f"FAILED: {group}")

# Preserve the original group order before handing results downstream.
per_group_results = {g: per_group_results[g] for g in groups if g in per_group_results}

if not per_group_results:
    raise RuntimeError("No per-group synthesis results were produced; stopping before downstream cells.")


Done: __gang_violence_exposure__
Done: __discipline_exclusion_absence__
Done: __direct_attendance_absence_terms__
Done: __community_violence_safety__
Done: __alternative_school_justice_adjacent__
Done: __mentoring_protective_support__
Done: __justice_involved_family__


---
## Step 7 — Structured Insight Extraction

Assembles the cross-group and per-group synthesis texts and asks the model to
return a structured JSON object with `key_insights` (cross-group) and `by_group`
(per-group, one list per group value). The response is validated, normalized,
and written to `insights_candidates.json` as a pre-verification snapshot.

In [12]:
# ── EXTERNAL-FACING INSIGHTS — structured JSON call ───────────────────────
# Ask the model for a single JSON object, then normalize and validate it.

OUTPUT_GROUP_KEY = "by_group"
REQUIRED_GROUP_VALUES = list(per_group_results.keys())

if not REQUIRED_GROUP_VALUES:
    raise RuntimeError("No synthesized group results are available; cannot build external-facing insights.")

synthesis_parts = []
if synthesis_cross:
    synthesis_parts.append(f"=== CROSS-GROUP ANALYSIS ===\n{synthesis_cross}")
for group_value, text in per_group_results.items():
    if text:
        synthesis_parts.append(f"=== {group_value} ===\n{text}")
synthesis_input = "\n\n".join(synthesis_parts)

INSIGHTS_SYSTEM = (
    "You are a senior program analyst at DonorsChoose. "
    "You turn structured internal analysis into clear, evidence-grounded insight objects for review by "
    "foundation leaders, corporate partners, policymakers, executives, and major donors. "
    "You return ONLY valid JSON. No preamble, no explanation, no markdown fences.\n\n"

    "Your job is not to write dramatic reveals. Your job is to state what the evidence supports, "
    "explain the classroom or student-facing mechanism, and make clear how far the claim should be generalized.\n\n"

    "Evidence discipline:\n"
    "- Use the structured synthesis as the primary evidence source.\n"
    "- Treat synthesized Direct evidence, Interpretation, Scope, and Supporting topics as evidence units.\n"
    "- Every insight must be grounded in the strongest matching supporting topics.\n"
    "- Do not infer broad support if the synthesis only gives narrow support.\n"
    "- Prefer fewer well-matched source_topics over padding with weak ones.\n"
    "- Do not merge adjacent synthesized findings unless they clearly support the same final insight.\n"
    "- Do not use project-topic support counts as proof that every project expresses the full final claim.\n\n"

    "Insight selection:\n"
    "- A valid insight may be a plain mechanism, a meaningful split, a boundary, a sparse emerging signal, or a practical funding/reporting implication.\n"
    "- Do not force every insight into a looks-like-X-but-is-Y structure.\n"
    "- Do not manufacture a reader mistake or surface-level misread.\n"
    "- Avoid observations that merely restate what the group label already implies.\n"
    "- Prefer the smallest non-overlapping set of insights that captures the strongest supported patterns.\n\n"

    "Substrate rule:\n"
    "- Food, hygiene, clothing, seating, storage, sensory tools, calm spaces, headphones, basic supplies, and generic devices are common classroom substrate.\n"
    "- Do not promote common substrate as a main insight unless the evidence shows a mechanism specific to the current group or cross-group pattern.\n"
    "- If substrate appears only as generic classroom operating support, mention it only as evidence, background, or scope.\n\n"

    "DonorsChoose voice:\n"
    "- Phrase findings around students, classrooms, or classroom conditions when that is natural and evidence-faithful.\n"
    "- Use teachers as the subject when the evidence is genuinely about what teachers request, build, adapt, manage, or describe.\n"
    "- Do not write awkward agentless prose just to avoid teacher-first phrasing.\n"
    "- The implication should connect back to students or classroom conditions when possible, but the sentence subject should be whatever makes the claim clearest.\n"
    "- Write directly, warmly, and concretely. Warmth should come from classroom materials, routines, and student conditions, not emotional adjectives.\n"
    "- Do not use named places.\n"
    "- Do not use em dashes.\n"
    "- Do not use jargon or corporate boilerplate.\n"
    "- Do not use phrases such as 'not just X,' 'at its core,' 'this speaks to,' 'it is clear that,' 'leverage,' 'utilize,' 'facilitate,' or 'impactful' as a standalone adjective.\n\n"

    "Scope rules:\n"
    "- Use cautious scope language for sparse, sensitive, or narrow groups.\n"
    "- Use phrases such as 'in this slice,' 'among these projects,' or 'a small but coherent signal' when support is limited.\n"
    "- Do not use trend language such as 'growing,' 'rising,' 'increasing,' or 'more often' unless the supplied evidence includes an explicit time comparison.\n\n"

    "Output rules:\n"
    "- Return ONLY valid JSON.\n"
    "- Use exactly the required schema and fields.\n"
    "- Do not add extra fields.\n"
    "- Do not include markdown fences, commentary, or explanation outside the JSON object."
)

example_source_topics = []
example_rows = (
    labels_df[[GROUPBY_FIELD, "topic_id"]]
    .drop_duplicates()
    .head(3)
    .to_dict("records")
)

for row in example_rows:
    example_source_topics.append(f"{row[GROUPBY_FIELD]}|{int(row['topic_id'])}")

if not example_source_topics:
    for i, gv in enumerate(REQUIRED_GROUP_VALUES[:2], start=1):
        example_source_topics.append(f"{gv}|{i}")

example_source_topics_json = json.dumps(example_source_topics, ensure_ascii=False)

INSIGHTS_PROMPT = f"""
You are given a structured synthesis of classroom project request patterns.

The synthesis is grouped by the field "{GROUPBY_FIELD}" ({GROUP_DESCRIPTION}).

It contains:
- cross-group findings
- within-group findings
- important distinctions
- boundaries or non-central signals
- explicit Supporting topics lines in the format <group_value>|<topic_id>

Structured synthesis:
{synthesis_input}

Your task:
Produce an evidence-grounded insights document for review by external-facing DonorsChoose audiences:
foundation leaders, corporate partners, policymakers, executives, and major donors.

The best insights should help a reader understand what students and classrooms need, what school-day conditions are shaping those needs, and what funding or reporting implications follow. Do not force surprise. A plain, well-scoped finding is better than a dramatic reveal.

WORKING METHOD:
1. Read the structured synthesis as a set of evidence units, not just as prose.
2. Identify plausible candidate cross-group insights and group-specific insights.
3. For each candidate, separate:
   - direct evidence: what the topics show
   - interpretation: what role that evidence appears to play
   - scope: how far the claim should be generalized
4. Remove or merge any candidate whose core implication materially overlaps another candidate.
5. Keep only the strongest non-overlapping set.
6. Then generate the final JSON.

SELECTION CRITERIA:
- Prioritize insights with a clear classroom or student-facing mechanism.
- Prioritize insights that matter for funding strategy, partnership design, policy interpretation, reporting, or external messaging.
- Current K-12 dynamics such as AI use in classrooms, post-pandemic recovery, technology continuity, cellphone management, transition planning, ESSER-cliff budget pressure, staffing strain, attendance, safety, and mental health are valuable only when directly supported by the synthesis.
- Avoid observations that merely restate what the group label already implies.
- Avoid methodological commentary and data-practitioner-only insights.
- Do not promote common classroom substrate as a main insight unless the evidence shows a mechanism specific to this group or pattern.
- Do not use trend language unless explicit time-comparison evidence is supplied.
- A boundary or non-central signal may become an insight if it prevents overgeneralization, distinguishes a sparse emerging topic, or clarifies what the category should not be used to claim.

DISTINCTNESS RULE:
- Each key insight must have a distinct core implication.
- Before finalizing "key_insights", check whether any two candidate key insights share the same mechanism, same reader takeaway, or same funding/strategy implication.
- If they do, keep the stronger one or merge them into one sharper insight.
- Do not return multiple key insights that are different phrasings or slight extensions of the same underlying point.

MERGE RULE:
- Combine synthesized findings into one final insight only when they clearly support the same interpretation of classroom demand.
- Shared product domain, broad similarity, or adjacent classroom context is not enough by itself.
- If a broader candidate and a narrower, more decision-useful candidate are both available, prefer the narrower one.
- Do not split one broad pattern into multiple final insights unless each split changes what a reader should conclude.

COUNT + COVERAGE REQUIREMENTS:
- Return between {CROSS_MIN_INSIGHTS} and {CROSS_MAX_INSIGHTS} cross-group insights in "key_insights".
- Treat {CROSS_MIN_INSIGHTS} as the default target.
- Start by asking whether the strongest {CROSS_MIN_INSIGHTS} key insights already capture the cross-group story.
- Only add more than {CROSS_MIN_INSIGHTS} if an additional insight is clearly distinct, non-overlapping, strongly supported, and would materially change what the reader learns.
- Never add an insight just because room remains before {CROSS_MAX_INSIGHTS}.
- Prefer the smallest non-overlapping set within the allowed range that captures the strongest supported patterns.

- "{OUTPUT_GROUP_KEY}" must include every group value in this exact list:
  {json.dumps(REQUIRED_GROUP_VALUES, ensure_ascii=False)}

- Return between {PER_GROUP_MIN_INSIGHTS} and {PER_GROUP_MAX_INSIGHTS} strongest defensible insights for each group value above.
- Treat {PER_GROUP_MIN_INSIGHTS} as the default target for each group.
- Start by asking whether the strongest {PER_GROUP_MIN_INSIGHTS} insights already capture the group's main internal story.
- Only add more than {PER_GROUP_MIN_INSIGHTS} when the additional insight introduces a clearly different mechanism, not just a narrower example, subcase, or rephrasing.
- Do not omit group values silently.
- Prefer slightly weaker but still defensible coverage over omission.
- Keep by-group insights focused on the group's dominant internal mechanisms and avoid repeating the same implication in multiple phrasings.

INSIGHT STRUCTURE:
For every insight, use exactly these fields:

- title:
  A short, plain mechanism statement.
  Usually center students, classrooms, classroom conditions, teacher actions, or school-day operations, whichever is clearest and most evidence-faithful.
  Do not write a slogan.
  Do not force a contrast frame.
  Do not use "really about," "not just," "more than," "hidden," or "easy to miss."

- finding:
  1-3 sentences explaining the pattern in direct, concrete language.
  State what the evidence supports.
  Phrase findings around students, classrooms, or classroom conditions when natural and evidence-faithful.
  Use teachers as the subject when the evidence is genuinely about what teachers request, build, adapt, manage, or describe.
  Do not write awkward agentless prose just to avoid teacher-first phrasing.
  Match length to the complexity of the signal. Do not pad simple findings.

- evidence_basis:
  1-2 sentences naming the concrete evidence behind the finding:
  materials, classroom routines, project contexts, repeated terms, student conditions, or topic descriptions.
  This should make clear what the interpretation rests on.
  Do not simply repeat the category name.

- scope_or_caveat:
  1 sentence explaining where the claim is strongest, narrowest, or most limited.
  If there is no major caveat, state the specific group, mechanism, or evidence base where the pattern is strongest.
  Do not write generic filler such as "This pattern may not apply everywhere."

- why_it_matters:
  1 sentence explaining the practical implication for interpretation, funding, reporting, partnership strategy, or policy understanding.
  This field is not a call to action and should not be donor-marketing copy.

- source_topics:
  A list of the strongest grounding topics for the insight.
  Use only topics that directly support the claim.
  Usually 2-4 topics is enough, but do not force a fixed count.
  Prefer fewer strong matches over weak padding.
  Required format:
    - each item must be a string
    - each string must be exactly "<group_value>|<topic_id>"
    - group_value must exactly match one of the group values shown in the synthesis
    - topic_id must be the integer topic number for that group
  Example for this run: {example_source_topics_json}

SOURCE_TOPIC RULES:
- Prefer source_topics that are explicitly named in Supporting topics lines.
- If a synthesized finding names multiple supporting topics but only some directly support your final claim, include only the strongest matches.
- If a topic is illustrative but not necessary to support the claim, leave it out.
- Do not use labels, prose, section names, or invented group names in place of group_value.
- source_topics must be a list of strings only, not objects.
- Do not include any extra fields beyond: title, finding, evidence_basis, scope_or_caveat, why_it_matters, source_topics.

STYLE:
- Direct, warm, concrete, and human.
- Not academic.
- Not corporate boilerplate.
- Not donor-marketing fluff.
- No named places.
- No em dashes.
- No mention of topics, models, clusters, prompts, or analytical machinery in title, finding, evidence_basis, scope_or_caveat, or why_it_matters.
- Do not manufacture a "why this is easy to miss" explanation.
- Do not use "may suggest" or "could indicate," but do use scope language when support is narrow.
- No filler.

VALIDITY RULES:
- Return valid JSON only.
- The response is incomplete if any required group value is missing from "{OUTPUT_GROUP_KEY}".
- Do not return markdown fences or explanatory text.
- Every source_topics item must follow the exact required format.
- Every insight object must contain exactly these six fields:
  title, finding, evidence_basis, scope_or_caveat, why_it_matters, source_topics

Return a JSON object with exactly this structure:
{{
  "key_insights": [/* {CROSS_MIN_INSIGHTS}-{CROSS_MAX_INSIGHTS} insight objects */],
  "{OUTPUT_GROUP_KEY}": {{
    "<group value>": [/* {PER_GROUP_MIN_INSIGHTS}-{PER_GROUP_MAX_INSIGHTS} insight objects */],
    ...
  }}
}}
""".strip()

resp = client.chat.completions.create(
    model=MODEL_SYNTHESIS,
    messages=[
        {"role": "system", "content": INSIGHTS_SYSTEM},
        {"role": "user", "content": INSIGHTS_PROMPT},
    ],
    response_format={"type": "json_object"},
)

raw_json = strip_json_fences(resp.choices[0].message.content)
insights_data = json.loads(raw_json)

if not isinstance(insights_data, dict):
    raise ValueError("Top-level insights response is not a JSON object.")

raw_key = insights_data.get("key_insights", [])
raw_by_group = insights_data.get(OUTPUT_GROUP_KEY, {})

if not isinstance(raw_key, list):
    raise ValueError("key_insights is not a list.")
if not isinstance(raw_by_group, dict):
    raise ValueError(f"{OUTPUT_GROUP_KEY} is not an object.")

normalized = {
    "key_insights": [
        normalize_insight(i, required_group_values=REQUIRED_GROUP_VALUES)
        for i in raw_key
    ],
    OUTPUT_GROUP_KEY: {},
}

missing_group_values = [g for g in REQUIRED_GROUP_VALUES if g not in raw_by_group]
if missing_group_values:
    raise ValueError(f"Model omitted required group values: {missing_group_values}")

extra_group_values = [g for g in raw_by_group.keys() if g not in REQUIRED_GROUP_VALUES]
if extra_group_values:
    print(f"WARNING: extra group values returned and ignored: {extra_group_values}")

for group_value in REQUIRED_GROUP_VALUES:
    items = raw_by_group.get(group_value, [])
    if not isinstance(items, list):
        raise ValueError(f"{OUTPUT_GROUP_KEY}['{group_value}'] is not a list.")
    normalized[OUTPUT_GROUP_KEY][group_value] = [
        normalize_insight(i, required_group_values=REQUIRED_GROUP_VALUES)
        for i in items
    ]

insights_data = normalized

if not (CROSS_MIN_INSIGHTS <= len(insights_data["key_insights"]) <= CROSS_MAX_INSIGHTS):
    raise ValueError(
        f"Expected between {CROSS_MIN_INSIGHTS} and {CROSS_MAX_INSIGHTS} key insights, "
        f"got {len(insights_data['key_insights'])}"
    )

bad_group_counts = {
    group_value: len(items)
    for group_value, items in insights_data[OUTPUT_GROUP_KEY].items()
    if not (PER_GROUP_MIN_INSIGHTS <= len(items) <= PER_GROUP_MAX_INSIGHTS)
}
if bad_group_counts:
    raise ValueError(
        f"Expected between {PER_GROUP_MIN_INSIGHTS} and {PER_GROUP_MAX_INSIGHTS} insights per group value, "
        f"got: {bad_group_counts}"
    )

HARD_STYLE_PATTERNS = [
    r"—",
    r"\bnot just\b",
    r"\bat its core\b",
    r"\bthis speaks to\b",
    r"\bit is clear that\b",
    r"\bperhaps most importantly\b",
    r"\bin a world where\b",
]

SOFT_STYLE_PATTERNS = [
    r"\bleverage\b",
    r"\butilize\b",
    r"\bfacilitate\b",
    r"\bimpactful\b",
    r"\bstakeholders\b",
    r"\bend users\b",
    r"\bbeneficiaries\b",
    r"\ba significant number of\b",
]

def _pattern_hits(text, patterns):
    return [p for p in patterns if re.search(p, text, flags=re.IGNORECASE)]

def _iter_all_insights_for_style(data, output_group_key):
    for item in data.get("key_insights", []):
        yield "key_insights", None, item
    for group_value, items in data.get(output_group_key, {}).items():
        for item in items:
            yield output_group_key, group_value, item

for section, group_value, item in _iter_all_insights_for_style(insights_data, OUTPUT_GROUP_KEY):
    combined = " ".join(
        str(item.get(k, "") or "")
        for k in ["title", "finding", "evidence_basis", "scope_or_caveat", "why_it_matters"]
    )

    hard_hits = _pattern_hits(combined, HARD_STYLE_PATTERNS)
    soft_hits = _pattern_hits(combined, SOFT_STYLE_PATTERNS)

    if hard_hits:
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "INSIGHT_HARD_STYLE_VIOLATION",
            "Insight contains hard-banned DonorsChoose style pattern",
            context={
                "section": section,
                "group_value": group_value,
                "title": item.get("title", ""),
                "violations": hard_hits,
            },
        )
        print(f"HARD STYLE WARNING: {item.get('title', '')[:80]} | {hard_hits}")

    if soft_hits:
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "INSIGHT_SOFT_STYLE_WARNING",
            "Insight contains soft-flagged DonorsChoose style pattern",
            context={
                "section": section,
                "group_value": group_value,
                "title": item.get("title", ""),
                "violations": soft_hits,
            },
        )
        print(f"SOFT STYLE WARNING: {item.get('title', '')[:80]} | {soft_hits}")

insights_candidates_for_save = {
    "key_insights": [
        project_insight_for_saved_candidates(i)
        for i in insights_data["key_insights"]
    ],
    OUTPUT_GROUP_KEY: {
        group_value: [
            project_insight_for_saved_candidates(i)
            for i in items
        ]
        for group_value, items in insights_data[OUTPUT_GROUP_KEY].items()
    },
}

# Mid-cell write required by spec: post-normalize_insight(), pre-verification.
write_json(OUT("insights", "insights_candidates.json"), insights_candidates_for_save)


PosixPath('/Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/safety_justice/2026-05-14/20260514_210306_safety_justice_0c79fb23/insights/insights_candidates.json')

---
## Step 8 — Topic Verification

Verifies that each claimed source topic genuinely supports its insight. Topics
that are too broad, indirect, or adjacent are dropped.

Verification strategy:
- **Cross-group insights** — always verified.
- **By-group insights** — verified only when they cite ≥ `by_group_min_source_topics`
  source topics; narrow local findings with 1–2 cited topics pass through without
  an additional API call.

In [13]:
# ── Topic verification ─────────────────────────────────────────────────────
# Verify source-topic grounding for final insights.
# Strategy:
# - always verify key_insights
# - verify by-group insights only when they cite 3+ source topics
#   (broad claims are more likely to overclaim support than narrow local ones)

VERIFY_SYSTEM = (
    "You are validating whether cited topics directly support an insight. "
    "Be strict. Keep a topic only if its label and description directly support "
    "the insight's title, finding, and evidence_basis. "
    "Ignore any practical implication or downstream strategy that is not part of the finding. "
    "If support is broad, indirect, adjacent, generic substrate, or only loosely related, drop it. "
    "Prefer false negatives to false positives. "
    "Return only valid JSON."
)

VERIFY_BY_GROUP_MIN_SOURCE_TOPICS = VERIFY_CFG["by_group_min_source_topics"]

# Always verify cross-group/key insights
insights_data["key_insights"], key_verify_stats = _verify_insight_list(
    insights_data.get("key_insights", []),
    labels_df=labels_df,
    groupby_field=GROUPBY_FIELD,
    required_group_values=REQUIRED_GROUP_VALUES,
    client=client,
    model_verify=MODEL_VERIFY,
    system_prompt=VERIFY_SYSTEM,
    warnings_path=WARNINGS_PATH,
    min_source_topics_to_verify=1,
)

group_verify_stats = {}
for cat in insights_data.get(OUTPUT_GROUP_KEY, {}):
    verified_items, stats = _verify_insight_list(
        insights_data[OUTPUT_GROUP_KEY][cat],
        labels_df=labels_df,
        groupby_field=GROUPBY_FIELD,
        required_group_values=REQUIRED_GROUP_VALUES,
        client=client,
        model_verify=MODEL_VERIFY,
        system_prompt=VERIFY_SYSTEM,
        warnings_path=WARNINGS_PATH,
        min_source_topics_to_verify=VERIFY_BY_GROUP_MIN_SOURCE_TOPICS,
    )
    insights_data[OUTPUT_GROUP_KEY][cat] = verified_items
    group_verify_stats[cat] = stats

total_group_changed = sum(v["changed_count"] for v in group_verify_stats.values())
total_group_zeroed = sum(v["dropped_to_zero_count"] for v in group_verify_stats.values())
total_group_topics_before = sum(v["topics_before"] for v in group_verify_stats.values())
total_group_topics_after = sum(v["topics_after"] for v in group_verify_stats.values())

print("Verification complete.")
print(
    f"Key insights: {key_verify_stats['insight_count']} insights | "
    f"{key_verify_stats['changed_count']} changed | "
    f"{key_verify_stats['dropped_to_zero_count']} dropped to zero topics | "
    f"{key_verify_stats['topics_before']} -> {key_verify_stats['topics_after']} source topics"
)
print(
    f"By-group insights: {sum(v['insight_count'] for v in group_verify_stats.values())} insights | "
    f"{total_group_changed} changed | "
    f"{total_group_zeroed} dropped to zero topics | "
    f"{total_group_topics_before} -> {total_group_topics_after} source topics | "
    f"verified only when source_topics >= {VERIFY_BY_GROUP_MIN_SOURCE_TOPICS}"
)

Adjusted source_topics: Calm spaces and coping tools help students regulate and return to learning | 5 -> 0
Adjusted source_topics: Justice- and violence-themed materials support structured discussion and express | 5 -> 4
Adjusted source_topics: Student expression creates a structured response to violence-related themes | 3 -> 2
Adjusted source_topics: Classroom health supports protect attendance from illness-related barriers | 3 -> 2
Verification complete.
Key insights: 6 insights | 2 changed | 1 dropped to zero topics | 26 -> 20 source topics
By-group insights: 24 insights | 2 changed | 0 dropped to zero topics | 56 -> 54 source topics | verified only when source_topics >= 3


---
## Step 9 — Evidence Tables

Expands each verified insight into a flat summary row (`insights_flat.csv`) and
a per-topic support detail row (`insight_topic_support.csv`). Projects are ranked
by combined topic_share across all of an insight's verified topics so Looker
links surface the most representative essays.

In [14]:
# ── VERIFIED INSIGHT SUPPORT TABLES ────────────────────────────────────────

assert "project_topic_bridge_df" in dir() and not project_topic_bridge_df.empty, (
    "project_topic_bridge_df missing — ensure the bridge-build cell ran successfully"
)

BRIDGE_LOOKUP = build_bridge_lookup(project_topic_bridge_df)

label_index = build_label_index(
    labels_df,
    groupby_field=GROUPBY_FIELD,
    warnings_path=WARNINGS_PATH,
)

insights_flat_df, insight_topic_support_df = build_verified_insight_tables(
    insights_data,
    OUTPUT_GROUP_KEY,
    groupby_field=GROUPBY_FIELD,
    bridge_lookup=BRIDGE_LOOKUP,
    label_index=label_index,
    run_id=RUN_ID,
    # csv_max_ids_per_insight controls insight row density; see params.yaml.
    top_project_id_limit=CSV_MAX_IDS_PER_INSIGHT,
)

insights_flat_df.to_csv(OUT("insights", "insights_flat.csv"), index=False)
insight_topic_support_df.to_csv(
    OUT("insights", "insight_topic_support_candidates.csv"),
    index=False,
)

print(f"Candidate insights summarized: {len(insights_flat_df):,}")

Candidate insights summarized: 30


---
## Step 10 — Packaging, Dedupe & Topline Selection

Three sequential operations:

1. **Packaging** — apply quality threshold gates. Insights that don't clear
   `min_verified_topic_count`, `min_supporting_project_count`, and
   `min_mean_topic_share` are rejected.
2. **Deduplication** — deterministic overlap-based dedup within pair types.
3. **Topline selection** — assigns `report_section` to each accepted insight:
   `main_cross` (top N by quality rank), `main_by_group` (top 1 per group),
   `appendix_cross`, `appendix_by_group`.

In [15]:
# ── PACKAGING + DEDUPE + SIMPLE TOPLINE CUT ────────────────────────────────
# Business rule:
# - accept insights that clear threshold gates
# - remove obvious duplicates
# - then choose topline from the remaining pool
#
# Main section:
# - top N cross-category insights by:
#     1) supporting_project_count
#     2) verified_topic_count
#     3) mean_topic_share_all_verified_topics
# - top 1 by-group insight per category using the same ranking
#
# Appendix:
# - all other accepted insights

packaging = apply_deterministic_packaging(
    insights_flat_df,
    output_group_key=OUTPUT_GROUP_KEY,
    packaging_cfg=PACKAGING_CFG,
)

accepted_pack_df = packaging["accepted_df"]

print(f"Total insights before packaging: {len(insights_flat_df)}")
print(f"Accepted after thresholds: {len(accepted_pack_df)}")

curated_df, dedup_audit_df = dedupe_packaged_insights(
    accepted_pack_df,
    dedupe_cfg=DEDUPE_CFG,
)

curated_df = curated_df.copy()
curated_df["looker_url"] = curated_df["top_project_ids"].apply(
    lambda ids: build_looker_project_url(
        base_url=LOOKER_BASE_URL,
        project_ids=ids,
        filter_field=LOOKER_FILTER_FIELD,
        fields=LOOKER_FIELDS,
        limit=LOOKER_LIMIT,
        max_ids=LOOKER_ID_LIMIT,
    )
)

# main_min_verification_ratio gates eligibility for main-section placement;
# sourced from params.yaml → analysis.packaging.main_min_verification_ratio.
curated_df = assign_topline_sections_simple(
    curated_df,
    output_group_key=OUTPUT_GROUP_KEY,
    main_cross_limit=PACKAGING_CFG["main_cross_limit"],
    main_min_verification_ratio=MAIN_MIN_VERIFICATION_RATIO,
)

main_cross_df = curated_df[curated_df["report_section"] == "main_cross"].copy()
main_by_group_df = curated_df[curated_df["report_section"] == "main_by_group"].copy()
appendix_df = curated_df[
    curated_df["report_section"].isin(["appendix_cross", "appendix_by_group"])
].copy()

accepted_ids = set(curated_df["insight_id"])
rejected_ids = set(insights_flat_df["insight_id"]) - accepted_ids

# export project_id list by insight
insight_project_df = (
    insight_topic_support_df[
        insight_topic_support_df["insight_id"].isin(accepted_ids)
    ][["insight_id", "group_value", "topic_id"]]
    .merge(
        project_topic_bridge_df[["project_id", GROUPBY_FIELD, "topic_id"]],
        left_on=["group_value", "topic_id"],
        right_on=[GROUPBY_FIELD, "topic_id"],
        how="left",
    )
    [["insight_id", "project_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
insight_project_df.to_csv(OUT("insights", "insight_project_bridge.csv"), index=False)

# ── EXPORT: insight-with-text + sampled essays ────────────────────────────
# Companion to insight_project_bridge.csv, but with:
#   1) insight title / finding / evidence fields instead of insight_id only
#   2) essay text instead of project_id only
#   3) capped at N projects per insight (random sample, fixed seed)
from utils import load_essay_snippet_lookup

ESSAY_SAMPLE_PER_INSIGHT = 20
ESSAY_MAX_CHARS = 1200  # bump if you want fuller essays per row

# 1) random-sample up to N projects per insight (deterministic via seed)
sampled_pairs_df = (
    insight_project_df
    .groupby("insight_id", group_keys=False)
    .apply(
        lambda g: g.sample(
            n=min(ESSAY_SAMPLE_PER_INSIGHT, len(g)),
            random_state=42,
        )
    )
    .reset_index(drop=True)
)

# 2) look up essay text for just the sampled project IDs
import re

ACTUAL_TEXT_COL = "tokens"
needed_ids = set(sampled_pairs_df["project_id"].unique())
essay_lookup: dict = {}

for fpath in sorted((ROOT / "DATA").glob("project_essays*.csv")):
    if len(essay_lookup) == len(needed_ids):
        break
    for chunk in pd.read_csv(
        fpath, usecols=["project_id", ACTUAL_TEXT_COL], chunksize=200_000
    ):
        sub = chunk[chunk["project_id"].isin(needed_ids - set(essay_lookup))]
        for _, row in sub.iterrows():
            text = re.sub(r"\s+", " ", str(row.get(ACTUAL_TEXT_COL, "") or "")).strip()
            if text:
                essay_lookup[row["project_id"]] = text[:ESSAY_MAX_CHARS]
        if len(essay_lookup) == len(needed_ids):
            break
sampled_pairs_df["essay_text"] = (
    sampled_pairs_df["project_id"].map(essay_lookup).fillna("")
)

# 3) attach the polished insight text from curated_df
insight_text_df = curated_df[[
    "insight_id", "title", "finding", "evidence_basis",
    "scope_or_caveat", "why_it_matters",
    "category_bucket", "report_section",
]].drop_duplicates(subset=["insight_id"])

insight_project_text_df = (
    sampled_pairs_df
    .merge(insight_text_df, on="insight_id", how="left")
    [[
        "insight_id", "report_section", "category_bucket",
        "title", "finding", "evidence_basis",
        "scope_or_caveat", "why_it_matters",
        "project_id", "essay_text",
    ]]
    .sort_values(["report_section", "insight_id", "project_id"])
    .reset_index(drop=True)
)

insight_project_text_df.to_csv(
    OUT("insights", "insight_project_text_sample.csv"), index=False
)
print(
    f"Wrote insight_project_text_sample.csv: "
    f"{len(insight_project_text_df):,} rows across "
    f"{insight_project_text_df['insight_id'].nunique()} insights "
    f"(up to {ESSAY_SAMPLE_PER_INSIGHT} essays each)"
)

curated_df.to_csv(OUT('chart_data', 'curated_df.csv'), index=False)

print(f"Accepted before dedupe: {len(accepted_pack_df)}")
print(f"Dropped as obvious duplicates: {len(dedup_audit_df)}")
print(f"Accepted after dedupe: {len(curated_df)}")
print(f"Main cross-category selected: {len(main_cross_df)}")
print(f"Main by-group selected: {len(main_by_group_df)}")
print(f"Appendix selected: {len(appendix_df)}")
print(f"{len(insight_project_df):,} insight-project pairs across {insight_project_df['insight_id'].nunique()} insights")

Total insights before packaging: 30
Accepted after thresholds: 27


/var/folders/j3/jwjf6cwj7czdz1klxbhhjst80000gp/T/ipykernel_45073/3042952426.py:94: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Wrote insight_project_text_sample.csv: 540 rows across 27 insights (up to 20 essays each)
Accepted before dedupe: 27
Dropped as obvious duplicates: 0
Accepted after dedupe: 27
Main cross-category selected: 5
Main by-group selected: 7
Appendix selected: 15
24,956 insight-project pairs across 27 insights


---
## Step 11 — Final Outputs & Manifests

Persists all accepted and rejected outputs, builds the DOCX report, and writes
stage and pipeline manifests. Running this cell again after a partial run is
safe — all writes are deterministic.

In [16]:
# ── FINAL OUTPUTS + REPORTING ──────────────────────────────────────────────
# Persist accepted/rejected outputs, evidence exports, chart data, DOCX, and manifests.

structured = build_structured_from_curated(
    curated_df,
    output_group_key=OUTPUT_GROUP_KEY,
)

write_json(OUT("insights", "insights_structured.json"), structured)

rejected_df = insights_flat_df[
    ~insights_flat_df["insight_id"].isin(curated_df["insight_id"])
].copy()
write_json(
    OUT("insights", "rejected_insights.json"),
    rejected_df.to_dict(orient="records"),
)

insight_topic_support_df[
    insight_topic_support_df["insight_id"].isin(curated_df["insight_id"])
].to_csv(
    OUT("insights", "insight_topic_support.csv"),
    index=False,
)

chart_ready_insights_df = curated_df[
    [
        "run_id",
        "insight_id",
        "title",
        "section",
        "category_bucket",
        "supporting_project_count",
        "verified_topic_count",
        "mean_topic_share_all_verified_topics",
        "verification_ratio",
        "report_section",
        "report_order",
    ]
].rename(columns={"title": "theme"})

chart_ready_group_support_df = insight_topic_support_df[
    insight_topic_support_df["insight_id"].isin(curated_df["insight_id"])
].copy()

chart_ready_insights_df.to_csv(
    OUT("chart_data", "chart_ready_insights.csv"),
    index=False,
)
chart_ready_group_support_df.to_csv(
    OUT("chart_data", "chart_ready_group_support.csv"),
    index=False,
)

docx_path = OUT("reports", "trend_tracker_report.docx")
build_packaged_report_docx(
    structured=structured,
    output_path=docx_path,
    output_group_key=OUTPUT_GROUP_KEY,
    report_cfg=CFG["output"],
    project_count=len(df),
    run_id=RUN_ID,
)

groups_failed = sorted(set(NMF_GROUPS_FAILED) | set(LABELING_FAILED_GROUPS) | set(SYNTHESIS_FAILED_GROUPS))
eligible_groups = list(per_group_results.keys())
manifest_status = "failure" if groups_failed else "success"

stage_manifest_path = OUT("metadata", "stage_manifest_03_insights_generation.json")
finalize_stage_manifest(
    STAGE_MANIFEST,
    output_path=stage_manifest_path,
    status=manifest_status,
    input_artifacts=[
        artifact_meta(ROOT / "OUTPUTS/prepared/06_enriched.parquet", "enriched_parquet"),
        artifact_meta(ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json", "stage_manifest_01"),
        artifact_meta(ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json", "stage_manifest_02"),
    ],
    output_artifacts=[
        artifact_meta(COPIED_CONFIG_PATH, "resolved_params_yaml"),
        artifact_meta(OUT("analysis", "category_tfidf.csv"), "category_tfidf_csv"),
        artifact_meta(OUT("analysis", "nmf_topics.csv"), "nmf_topics_csv"),
        artifact_meta(OUT("analysis", "nmf_weights.csv"), "nmf_weights_csv"),
        artifact_meta(OUT("analysis", "project_topic_bridge.csv"), "project_topic_bridge_csv"),
        artifact_meta(OUT("analysis", "llm_topic_labels.json"), "llm_topic_labels_json"),
        artifact_meta(OUT("insights", "insights_candidates.json"), "insights_candidates_json"),
        artifact_meta(OUT("insights", "insights_structured.json"), "insights_structured_json"),
        artifact_meta(OUT("insights", "rejected_insights.json"), "rejected_insights_json"),
        artifact_meta(OUT("insights", "insights_flat.csv"), "insights_flat_csv"),
        artifact_meta(OUT("insights", "insight_topic_support.csv"), "insight_topic_support_csv"),
        artifact_meta(OUT("chart_data", "chart_ready_insights.csv"), "chart_ready_insights_csv"),
        artifact_meta(OUT("chart_data", "chart_ready_group_support.csv"), "chart_ready_group_support_csv"),
        artifact_meta(OUT("reports", "trend_tracker_report.docx"), "trend_tracker_report_docx"),
    ],
    row_counts={
        "input_projects": int(len(raw_df)),
        "filtered_projects": int(len(df)),
        "eligible_groups": int(len(eligible_groups)),
        "groups_skipped": int(len(NMF_GROUPS_SKIPPED)),
        "groups_failed": int(len(groups_failed)),
        "topics_generated": int(len(topics_df)),
        "insights_generated": int(len(insights_flat_df)),
        "insights_accepted": int(len(accepted_ids)),
        "insights_rejected": int(len(rejected_ids)),
    },
    key_params={
        "group_by": GROUPBY_FIELD,
        "filter_fields_key": FILTER_FIELDS_KEY,
        "min_group_projects": CFG["analysis"]["min_group_projects"],
        "topic_assignment_threshold": CFG["analysis"]["topic_assignment_threshold"],
        "model_labeling": MODEL_LABELING,
        "model_synthesis": MODEL_SYNTHESIS,
        "verification_config": VERIFY_CFG,
        "packaging_config": PACKAGING_CFG,
        "dedupe_config": DEDUPE_CFG,
    },
    warnings_path=WARNINGS_PATH,
)

pipeline_manifest_path = OUT("metadata", "pipeline_manifest.json")
build_pipeline_manifest(
    output_path=pipeline_manifest_path,
    run_id=RUN_ID,
    run_date=RUN_DATE,
    status=manifest_status,
    config_path=MANIFEST_CFG_PATH,
    group_by_field=GROUPBY_FIELD,
    filter_fields_key=FILTER_FIELDS_KEY,
    filter_spec_path=FILTER_SPEC_PATH,
    filter_summary_path=FILTER_SUMMARY_PATH,
    stage_manifest_paths=[
        ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json",
        ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json",
        stage_manifest_path,
    ],
    warnings_01_path=ROOT / "OUTPUTS/prepared/warnings_01.jsonl",
    warnings_02_path=ROOT / "OUTPUTS/enrichment/warnings_02.jsonl",
    warnings_03_path=WARNINGS_PATH,
    final_outputs={
        "insights_structured_json": str(OUT("insights", "insights_structured.json")),
        "rejected_insights_json": str(OUT("insights", "rejected_insights.json")),
        "insights_flat_csv": str(OUT("insights", "insights_flat.csv")),
        "insight_topic_support_csv": str(OUT("insights", "insight_topic_support.csv")),
        "chart_ready_insights_csv": str(OUT("chart_data", "chart_ready_insights.csv")),
        "chart_ready_group_support_csv": str(OUT("chart_data", "chart_ready_group_support.csv")),
        "trend_tracker_report_docx": str(OUT("reports", "trend_tracker_report.docx")),
    },
)

print(f"Accepted insights: {len(accepted_ids):,}")
print(f"Rejected insights: {len(rejected_ids):,}")
print(f"Docx saved → {docx_path}")
print(f"Stage manifest written → {stage_manifest_path}")
print(f"Pipeline manifest written → {pipeline_manifest_path}")

Accepted insights: 27
Rejected insights: 3
Docx saved → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/safety_justice/2026-05-14/20260514_210306_safety_justice_0c79fb23/reports/trend_tracker_report.docx
Stage manifest written → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/safety_justice/2026-05-14/20260514_210306_safety_justice_0c79fb23/metadata/stage_manifest_03_insights_generation.json
Pipeline manifest written → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/safety_justice/2026-05-14/20260514_210306_safety_justice_0c79fb23/metadata/pipeline_manifest.json
